## Object Class Implementation
 - Need to implement the DLA class with methods for simulating particle aggregation, animating the growth of the cluster, and analyzing its fractal properties. This includes defining the stickiness parameter, handling particle movement and attachment, and implementing box-counting methods for dimension analysis.

In [ ]:
import numpy as np
import random
import matplotlib.pyplot as plt
from   matplotlib.animation import FuncAnimation
from   matplotlib import animation, colors
from   IPython.display import HTML

class DLA:
    def __init__(self, stickiness, max_particles):
        """
        Initialization function:
            Common python convention is to use __init__ for initialization of class attributes.
            This function sets up the grid, initializes the cluster with a seed particle, and prepares parameters for the simulation.
        """
        # INITIALIZATION OF SIMULATION PARAMETERS
            # The grid size is determined based on the # of particles and the expected fractal dimension, ensuring enough space for growth without being excessively large.
            # I need to make the gird size a function of max_particles, because I don't know how big the grid should be for a million particles
        D                   = 1.65                              # Fractal dimension for DLA
        R                   = max_particles**(1/D)              # Expected radius of the cluster based on the number of particles and the fractal dimension
        self.grid_size      = int(2.5*(R) + 20)                 # Length and width of the square grid: (2*radius of cluster) + a buffer to prevent edge cases

        self.compass        = [(1,0),(-1,0),(0,1),(0,-1),(1,1),(1,-1),(-1,1),(-1,-1)] # 8 possible movement directions (including diagonals)
        self.center         = self.grid_size // 2               # Center of the grid
        self.max_steps      = self.grid_size * 10               # Maximum steps a particle can take before being considered "lost" (to prevent infinite wandering)
        self.stickiness     = stickiness                        # Probability of sticking when adjacent
        self.max_particles  = max_particles                     # Total number of particles to add to the cluster 

        # INITIALIZATION OF GRID
        self.grid = np.zeros((self.grid_size, self.grid_size))  # Create the square grid (matrix) 
        self.grid[self.center, self.center] = 1                 # Used floor division to ensure integer index. Put seed in the center of grid
        
        # INITIALIZATION OF CLUSTER
        self.cluster_particles  = [(self.center, self.center)]  # Creates a tuple list to store the coords of particles in the cluster
        self.cluster_radius = 1                                 # Initial radius of the cluster (distance from center to farthest particle)
        self.kill_radius    = (self.cluster_radius * 1.25) + 10 # Prevent infinite walking (particles wandering forever)

    def new_particle(self):
        hist, bin_centers = self.compute_angular_density()
        """
        Introduce a new particle on a circle around the cluster
        Use and angle determined by the random generator and the bias created by the angular density
        Use a radius that is larger than the current cluster radius
        """
        weights   = (1 - hist)**2                                   # Bias towards less dense angles by giving them higher weights. Squaring the term increases the bias effect.
        weights  /= np.sum(weights)                                 # Normalize weights to sum to 1 for valid probability distribution

        idx       = np.random.choice(len(bin_centers), p=weights)   # Randomly select an angle bin (10 deg region) based on the computed weights
        theta     = bin_centers[idx]                                # Get the central angle of the selected bin for spawning the new particle

        bin_width = bin_centers[1] - bin_centers[0]                 # In case the bin sizes change, calculate the width of the bin for next step
        theta    += random.uniform(-bin_width/2, bin_width/2)       # Add a random offset within the bin for the final angle

        r = min(self.cluster_radius + 5, self.center - 2)           # Starting position is outside the cluster, so the particle can diffuse
        x = int(self.center + r * np.cos(theta))                    # The int() function forces the resulting vector to be an integer value
        y = int(self.center + r * np.sin(theta))                    # Since we are using computers and walking particles on a lattice, we need ints
        
        return x, y

    def step(self, x, y):
        """
        I have learned that python has a great built-in "random" library
        This simple function simply takes the current pos, and randomly steps it
        """
        direction = self.compass[random.getrandbits(3)]
        return x + direction[0], y + direction[1]
    
    def walk(self):
        """
        Was initially apart of the simulation method, but I broke it into a seperate method
        This function simulates the random walk of a single particle until it either sticks to the cluster or wanders too far away.
        """
        x, y = self.new_particle()                              # Start a new particle
        steps = 0                                               # Step counter for ending loop (prevent infinite loops)

        while steps < self.max_steps:
            x, y =self.step(x, y)                               # Take a random step
            steps = steps + 1                                   # Increment step counter
            """ 
            If particle wonders too far away from cluster: (e.g. if it tries to leave the grid)
                Simulate it "leaving" and another particle "entering" 
                by breaking the loop and starting a new one with a new particle.
            """

            r = np.sqrt((x - self.center)**2 + (y - self.center)**2)
            if (r > self.kill_radius):                          # The spawn distance was radius*2... don't let it leave the grid, prevent edge cases
                x, y = self.new_particle()                      # Respawn a new particle, the code will automatically stop tracking the old one and start tracking the new one
                steps = 0                                       # Reset step counter for the new particle

            # Determine if the particle is adjacent to the cluster and should stick
            neighbors = self.neighbor_counter(x, y)                # Check if the particle is adjacent to the cluster
            if neighbors > 0:
                p_stick = 1 - (1 - self.stickiness)**neighbors  # Calculate the probability of sticking based on the number of adjacent cluster particles
                if random.random() < p_stick:                   # This is where stickiness/probability matters. btw, random.random() generates a float between 0 and 1
                    return x, y                                 # The particle stuck: end here

        # Else, return None, indicating the particle wandered without sticking
        return None

    def neighbor_counter(self, x, y):
        """
        Short function that checks surrounding positions for cluster particles.
        Might modify later to check for multiple contact points
        """
        surrounding_pts = [(1,0),(1,-1),(0,-1),(-1,-1),(-1,0),(-1,1),(0,1),(1,1)]   # Steps needed to get from curr_pos to all surrounding_pts
        neighbors = 0                                   # Counter to track how many cluster particles are adjacent (for potential future use)
        for dx, dy in surrounding_pts:                  # Quick loop through all surrounding points
            nx, ny = x + dx, y + dy
            if 0 <= nx < self.grid_size and 0 <= ny < self.grid_size:
                if self.grid[nx, ny] > 0:               # Check if any surrounding point is part of the cluster
                    neighbors = neighbors + 1
        return neighbors

    def update_cluster_radius(self, x, y):
        r = np.sqrt((x - self.center)**2 + (y - self.center)**2)    # Simple distance formula to find new rad
        if r > self.cluster_radius:                                 # Check if radius needs updated
            self.cluster_radius = r                                 # Update as needed   
            self.kill_radius    = (self.cluster_radius * 1.25) + 10 # Prevent infinite walking (particles wandering forever)

    def compute_angular_density(self, n_bins=36):
        """
        Compute angular density histogram of cluster particles. 
        Starts off with logic similar to analyze_fractal
        Returns:
            hist (normalized), bin_centers
        """
        positions = np.array(self.cluster_particles)    # Convert list of tuples to a 2D np.array for easy calculations
        dx = positions[:,0] - self.center               # Shift the x coordinates so that the center of the cluster is at (0,0) - makes it simple later
        dy = positions[:,1] - self.center               # Shift the y coordinates so that the center of the cluster is at (0,0)

        angles = np.arctan2(dy, dx)                     # This function returns the angle in radians. Range [-π, π]

        # Create histogram of angles
        hist, bin_edges = np.histogram(angles, bins=n_bins, range=(-np.pi, np.pi))      # np.histogram returns the count of angles in each bin (10 deg), and the edges of the bins. Range is set to cover all possible angles.
        hist = hist.astype(float)                       # Convert to float for normalization

        # Normalize to [0,1] for easy biasing
        if hist.max() > 0:
            hist /= hist.max()

        bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])    # Calculate the center of each bin for plotting
        return hist, bin_centers

    def get_active_region(self, whitespace=5):
        '''
        This function determines the bounding box of the active cluster region and returns a cropped view of the grid for visualization.
        It finds the min and max x and y coordinates of the cluster particles, adds a small whitespace buffer, and returns the corresponding slice of the grid. 
        This helps to focus the visualization on the region where the cluster is actively growing.

        Based on code in the Tips and Tricks
        '''
        xslice = np.where(np.sum(self.grid, axis=0))[0]
        yslice = np.where(np.sum(self.grid, axis=1))[0]

        if len(xslice) == 0 or len(yslice) == 0:
            return self.grid  # fallback (initial state)

        x0 = max(xslice[0] - whitespace, 0)
        x1 = min(xslice[-1] + whitespace + 1, self.grid_size)

        y0 = max(yslice[0] - whitespace, 0)
        y1 = min(yslice[-1] + whitespace + 1, self.grid_size)

        return self.grid[y0:y1, x0:x1]

    def analyze_fractal(self, box_sizes=None):
        """
        Computes and plots the capacity (box-counting) dimension.
        Returns the estimated fractal dimension.
        """

        def box_partitioning(cluster, box_size):                        # Helper function to count how many boxes of a given size are needed to cover the cluster
            length = cluster.shape[0]                                   # Get the size of the cluster grid (after cropping)
            count = 0                                                   # Counter for how many boxes have particles in them
            for i in range(0, length, box_size):                        # Loop through the grid in steps of box_size to create non-overlapping boxes
                for j in range(0, length, box_size):                    # ''
                    if np.any(cluster[i:i+box_size, j:j+box_size] > 0): # Check if there is at least one particle in the current box
                        count += 1                                      # If there is, increment the count of boxes needed to cover the cluster
            return count                                            

        # Use cropped cluster
        cluster = self.get_active_region()

        # Adaptive box sizes (powers of 2)
        if box_sizes is None:                                           # If no box sizes provided, use powers of 2 up to a limit based on cluster size
            max_exp = int(np.log2(cluster.shape[0]))                    # Largest box size is the largest power of 2 that fits within the cluster grid
            box_sizes = [2**i for i in range(1, max_exp - 1)]           # Create a list of box sizes (2, 4, 8, ..., up to max_exp-1) for the box-counting method

        total_count = [box_partitioning(cluster, size) for size in box_sizes] # Count the number of boxes needed for each box size

        # Convert to logs
        log_S = np.log(1/np.array(box_sizes))
        log_N = np.log(total_count)

        # Linear fit → capacity dimension
        m, b = np.polyfit(log_S, log_N, 1)          # np.polyfit performs a linear fit to the log-log data, returning the coefficients of the best fit line.
        capacity_dimension = m                      # The capacity dimension is the negative of the slope from the log-log plot

        # Plot (clean version)
        plt.figure(figsize=(6,4))
        plt.plot(log_S, log_N, 'o', label='Data')
        plt.plot(log_S, np.polyval([m, b], log_S), '-', label=f'Fit: m = {m:.3f}')

        plt.xlabel('-log(Box Size)')
        plt.ylabel('log(N Boxes)')
        plt.title('Box-Counting (Capacity) Dimension')
        plt.legend()
        plt.grid(True)
        plt.savefig("capacity_dimension.png", dpi=300, bbox_inches="tight")
        plt.show()

        return capacity_dimension
    
    def CD_vs_radius(self, radial_bins=100):
        """
        Computes the local capacity dimension as a function of radius:
            D(r) = d log M(r) / d log r
        where M(r) is the mass (particle count) inside radius r.
        """

        # --- Extract particle positions ---
        positions = np.array(self.cluster_particles)
        dx = positions[:, 0] - self.center
        dy = positions[:, 1] - self.center
        radii = np.sqrt(dx**2 + dy**2)

        max_r = np.max(radii)

        # --- Radial bins ---
        r_edges = np.linspace(1e-6, max_r, radial_bins)
        M_r = np.zeros_like(r_edges)

        # cumulative mass function
        for i, r in enumerate(r_edges):
            M_r[i] = np.sum(radii <= r)

        # --- log transform ---
        log_r = np.log(r_edges)
        log_M = np.log(M_r + 1e-12)  # avoid log(0)

        # --- local slope = dimension ---
        D_r = np.gradient(log_M, log_r)

        plt.figure(figsize=(6, 4))
        plt.plot(r_edges, D_r)
        plt.xlabel("Radius r")
        plt.ylabel("Local capacity dimension D(r)")
        plt.title("Capacity Dimension vs Radius")
        plt.grid(True)
        plt.savefig("CD_vs_radius_2.png", dpi=300, bbox_inches="tight")
        plt.show()

        return r_edges, D_r

    # This is the "new" main loop for simulating the DLA process
    # Call this function to animate the growth of the cluster in real time. It uses matplotlib's FuncAnimation to update the plot as new particles are added.
    def animate(self, interval=50):
        fig, ax = plt.subplots(figsize=(8,8))                           # Set up the figure and axis for animation  
        initial_view = self.get_active_region()                         # Get the initial view of the grid (focused on the cluster)

        # Dynamically adjust color scale limits based on the initial grid state to maintain good contrast in the visualization as the cluster grows
        nonzero = initial_view[initial_view > 0]                        # Get the non-zero values in the initial view to determine appropriate color scale limits
        if len(nonzero) > 0:                                            # If there are non-zero values, set vmin and vmax based on percentiles
            vmin = np.percentile(nonzero, 5)                            
            vmax = np.percentile(nonzero, 98)
        else:                                                           # If there are no non-zero values (e.g. at the very start), set default limits to avoid errors
            vmin, vmax = 1, 2 

        im = ax.imshow(initial_view, cmap='turbo',
                norm=colors.LogNorm(vmin=max(1, vmin), vmax= vmax), origin='lower')   # ax.imshow is used to display the grid as an image, with a logarithmic color scale to show the age of particles in the cluster (older particles are darker)
        
        ax.set_title('Diffusion Limited Aggregation Cluster Growth')    # Self explanatory
        ax.axis('off')                                                  # Self explanatory 
        ax.set_aspect('equal')                                          # Keep the aspect ratio of the plot equal to avoid distortion                                     
        cbar = plt.colorbar(im, ax=ax)                                  # Self explanatory
        cbar.set_label("Particle Deposition Order")                     # Self explanatory
        
        particles_added = 1                                             # From the seed

        def frame_gen():
            while particles_added < self.max_particles:
                yield particles_added

        # This function will be called for each frame of the animation. It adds a batch of particles to the cluster and updates the visualization.
        def update(frame):
            nonlocal particles_added                                    # 'nonlocal' allows me to modify the variable defined outside scope of the method 
            
            batch = max(5, int(0.01 * particles_added))                  # Dynamically adjust batch size based on the current number of particles - better visual 

            # Add multiple particles per frame (important for speed)
            added_this_frame = 0                                        # Counter to track how many particles added in this frame
            while added_this_frame < batch and particles_added < self.max_particles:
                result = self.walk()                                    # Simulate the random walk for a single particle and check if it sticks to the cluster                  
                if result is not None:                                  # If the particle stuck, update the grid and cluster information
                    x, y = result                                       # Unpack the coordinates of the stuck particle
                    self.grid[x, y] = particles_added                   # Update grid with the order of deposition (for color mapping)
                    self.cluster_particles.append((x, y))               # Add particle to cluster list
                    self.update_cluster_radius(x, y)                    # Update the cluster radius... if needed

                    particles_added += 1                                # Increment the count of particles in the cluster
                    added_this_frame += 1                               # Increment the count of particles added in this frame

            view = self.get_active_region()                             # Get the current view of the grid (focused on the cluster)
            im.set_data(view)                                           # Update the image data with the new grid state
            
            # \begin The following code dynamically adjusts the color scale limits. This helps maintain good contrast in the visualization as the cluster grows
            nonzero = view[view > 0]                                    # Get the non-zero values in the current view to determine appropriate color scale limits
            # Update the color scale limits based on the new grid state to maintain good contrast as the cluster grows
            if len(nonzero) > 0:
                vmin = np.percentile(nonzero, 5)
                vmax = np.percentile(nonzero, 98)
            else:
                vmin, vmax = 1, 2
            im.set_norm(colors.LogNorm(vmin=max(1, vmin), vmax=vmax))   # Update the color scale based on the new grid state
            # \end

            return [im]                                                 # Return the updated image for blitting

        # FuncAnimation will call the update function for each frame of the animation. This visualizes the growth process
        ani = animation.FuncAnimation(fig, update, frames=frame_gen, interval=interval, blit=True, repeat=False, cache_frame_data=False)
        plt.close(fig)                                                  # Prevent a random static plot from showing in Jupyter Notebook when the animation runs

        return ani      # Return the animation object for display in Jupyter Notebook
    

## Call the Simulation and Analyze Results
Now that the object class has been fully implemented, we can run the simulation and analyze the results. The code below creates a DLA cluster with a specified stickiness parameter and maximum number of particles, animates the growth of the cluster, and then analyzes its fractal properties using box-counting methods to estimate the capacity dimension. The animation will be saved as an MP4 file, and the plot of the capacity dimension analysis will be saved as a PNG file for later reference.

In [23]:
import time

# Simulation parameters + Object Creation
stickiness      = 0.15
max_particles   = 1000000
sim = DLA(stickiness, max_particles)                # Create simulation object, run the simulation, and plot the results

# Run the simulation and analysis and time it
start_time = time.perf_counter()                    # Start timer to measure how long the animation takes to run
ani = sim.animate()                                 # Animate the growth of the cluster in real time, adding 10 particles per frame for smoother visualization
ani.save("1e6.mp4", writer='ffmpeg', fps=30)
# capacity_dimension = sim.analyze_fractal()          # Analyze fractal properties after animation completes
# cd_vs_radius = sim.CD_vs_radius()                   # Analyze how the capacity dimension changes with radius after animation completes
end_time = time.perf_counter()                      # End timer after analysis completes


# print(f"Estimated Capacity Dimension: {capacity_dimension:.3f}") # Print the estimated fractal dimension after the animation completes
print(f"Total Time: {end_time - start_time:.2f} seconds")


KeyboardInterrupt: 